<h1><p><center style="background: linear-gradient(to right,white,green);color: transparent;font-family: Gabriola;color: navy;font-size: 210%;text-align: center;border-radius: 10px 70px">
   Web Scarping 

</center></p></h1>

In [ ]:
from bs4 import BeautifulSoup
import requests
import csv
from datetime import date
import pandas as pd


In [ ]:



def scrape_opencodez(last_page_number):
    # Lists to store data
    title_list = []
    description_list = []
    author_list = []
    year_list = []
    month_day_list = []
    post_link_list = []
    post_image_link_list = []

    for p in range(1, last_page_number + 1):
        url = f'https://www.opencodez.com/page/{p}'
        response = requests.get(url)
        bs = BeautifulSoup(response.content, 'html.parser')

        latest_posts = bs.select_one('div.latest-section')
        if not latest_posts:
            print(f"Page {p}: Latest section not found.")
            continue

        # Extract elements using select
        title_all = latest_posts.select('h2.title a')
        description_all = latest_posts.select('div.post-content.image-caption-format-1 p')
        author_all = latest_posts.select('span.theauthor a')
        date_all = latest_posts.select('span.thetime')
        post_image_link_all = latest_posts.select('div.featured-thumbnail img')

        # Check for consistent lengths
        lengths = [len(title_all), len(description_all), len(author_all),
                   len(date_all), len(post_image_link_all)]

        if len(set(lengths)) != 1:
            print(f"Page {p}: Skipping due to inconsistent element counts {lengths}")
            continue

        for i in range(lengths[0]):
            title_list.append(title_all[i].get_text().strip())
            description_list.append(description_all[i].get_text().strip())
            author_list.append(author_all[i].get_text().strip())

            # Split date into month/day and year
            date_text = date_all[i].get_text().strip().split(',')
            month_day_list.append(date_text[0].strip() if len(date_text) > 0 else "N/A")
            year_list.append(date_text[1].strip() if len(date_text) > 1 else "N/A")

            post_link_list.append(title_all[i].get('href'))
            post_image_link_list.append(post_image_link_all[i].get('src'))

        print(f"Scraping page number {p} finished.")

    # Create DataFrame
    df = pd.DataFrame({
        'Title': title_list,
        'Description': description_list,
        'Author': author_list,
        'Year': year_list,
        'MonthDay': month_day_list,
        'Link': post_link_list,
        'ImageLink': post_image_link_list
    })

    return df

# Example usage:
last_page_number = 6  # change as needed or for example pages 1,2,3,... 10
df_posts = scrape_opencodez(last_page_number)
df_posts.head()

Scraping page number 1 finished.
Scraping page number 2 finished.
Scraping page number 3 finished.
Scraping page number 4 finished.
Scraping page number 5 finished.
Scraping page number 6 finished.


,Title,Description,Author,Year,MonthDay,Link,ImageLink
0,Python Tutorial :#2 Python Vs Java,"As we know Python and Java, Both the languages...",Supriya,2022,Mar 10,https://www.opencodez.com/python/python-tutori...,https://www.opencodez.com/wp-content/uploads/2...
1,Python Tutorial : #1 Introduction to Python,Python is a most popular programming language ...,Supriya,2022,Feb 20,https://www.opencodez.com/python/python-tutori...,https://www.opencodez.com/wp-content/uploads/2...
2,Chain of Responsibility – Behavioral Design Pa...,The Chain of Responsibility Pattern comes unde...,Supriya,2020,Aug 1,https://www.opencodez.com/java/chain-of-respon...,https://www.opencodez.com/wp-content/uploads/2...
3,Flyweight Pattern – Structural Design Pattern,Flyweight pattern comes under Structural Desig...,Supriya,2020,May 20,https://www.opencodez.com/java/flyweight-patte...,https://www.opencodez.com/wp-content/uploads/2...
4,What Is Full Stack QA or Tester? 4 Steps Guide...,We have always heard the term full-stack devel...,Shilpa,2020,May 18,https://www.opencodez.com/software-testing/bec...,https://www.opencodez.com/wp-content/uploads/2...


In [ ]:
df.to_csv("OpenCodes_Posts.csv", index = False)